**optimized version**

In [1]:
# FILE: 00_Preprocess_Optimized.ipynb

import os
import torch
import cv2
import numpy as np
from tqdm import tqdm

# --- CONFIGURATION ---
SOURCE_FOLDER = "data/UCF50"
DEST_FOLDER = "data/UCF50_Tensors"
IMG_SIZE = 112
SEQUENCE_LENGTH = 16

os.makedirs(DEST_FOLDER, exist_ok=True)

def preprocess_video_optimized(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    try:
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames == 0: return None
        
        if total_frames < SEQUENCE_LENGTH:
            indices = list(range(total_frames))
        else:
            indices = np.linspace(0, total_frames - 1, SEQUENCE_LENGTH).astype(int)

        for i in range(total_frames):
            ret, frame = cap.read()
            if not ret: break
            if i in indices:
                frame = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
                # Keep as BGR or RGB? RGB is better for models.
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)
                if len(frames) == SEQUENCE_LENGTH: break
    finally:
        cap.release()

    frames = np.array(frames)
    
    # Pad if too short
    if len(frames) < SEQUENCE_LENGTH:
        padding = np.zeros((SEQUENCE_LENGTH - len(frames), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        if len(frames) > 0:
            frames = np.concatenate((frames, padding), axis=0)
        else:
            frames = padding

    # OPTIMIZATION: Keep as uint8 (0-255)
    # Shape: (T, H, W, C) -> (C, T, H, W)
    tensor = torch.ByteTensor(frames).permute(3, 0, 1, 2)
    return tensor

# --- MAIN EXECUTION ---
classes = sorted(os.listdir(SOURCE_FOLDER))
print(f"Starting optimized conversion for {len(classes)} classes...")

for cls in tqdm(classes):
    cls_path = os.path.join(SOURCE_FOLDER, cls)
    if not os.path.isdir(cls_path): continue
    
    save_path = os.path.join(DEST_FOLDER, cls)
    os.makedirs(save_path, exist_ok=True)
    
    videos = [v for v in os.listdir(cls_path) if v.endswith(('.avi', '.mp4'))]
    
    for vid in videos:
        out_name = vid.replace('.avi', '.pt').replace('.mp4', '.pt')
        out_file = os.path.join(save_path, out_name)
        
        if os.path.exists(out_file): continue 
        
        tensor = preprocess_video_optimized(os.path.join(cls_path, vid))
        if tensor is not None:
            torch.save(tensor, out_file)

print("\nProcessing Complete! Size should be ~3.8GB.")

Starting optimized conversion for 50 classes...


100%|██████████████████████████████████████████████████████████████████████████████████| 50/50 [10:02<00:00, 12.06s/it]


Processing Complete! Size should be ~3.8GB.
